# บทที่ 6 · ดึงข้อมูลจาก API และจัดการตารางด้วย pandas

**เรื่องสมมติของคอร์สนี้:** เราเป็นทีมข้อมูลของร้านค้าออนไลน์ (ร้านเดียวกับ session 3)
- ร้านจ่ายเงิน **ค่าโฆษณาออนไลน์** ทุกวัน ดูยอดได้จาก **API** ของระบบโฆษณา
- ร้านขายของบน **Shopee / Lazada / TikTok Shop** ด้วย แต่ละวัน marketplace ส่ง **ไฟล์ CSV** ยอดขายมาให้

API ที่ใช้: เปิดด้วย Docker แล้ว (ดู README) เอกสารอยู่ที่ http://localhost:8000/docs
**ตั้งใจให้ error สุ่ม 25%** เหมือนเว็บจริงที่บางครั้งไม่ว่าง

บทนี้: **requests · status code · API key · params · หลายหน้า · ลองใหม่ · pandas: สร้าง/อ่าน/กรอง/สรุป/รวมตาราง**

In [ ]:
import os

import requests
from dotenv import load_dotenv

load_dotenv()
API_URL = os.getenv("SHOP_API_URL", "http://127.0.0.1:8000")
API_KEY = os.getenv("SHOP_API_KEY", "class-demo-key")
print(API_URL)

## 6.1 ขอข้อมูลจากเว็บด้วย `requests.get`

API = หน้าเว็บสำหรับโปรแกรม: เราส่งคำขอ (request) ไป แล้วได้คำตอบ (response) กลับมาเป็น JSON

In [ ]:
response = requests.get(f"{API_URL}/health", timeout=10)     # timeout: ไม่รอเกิน 10 วินาที

print(response.status_code)     # 200 = สำเร็จ
print(response.text)            # คำตอบแบบข้อความ
print(response.json())          # แปลง JSON -> dict

| status | ความหมาย | ทำอย่างไร |
|---|---|---|
| 200 | สำเร็จ | ใช้ข้อมูล |
| 400 / 422 | เราส่งค่าผิด | แก้โค้ด (ลองใหม่ไม่ช่วย) |
| 401 / 403 | ไม่มีสิทธิ์ / key ผิด | แก้ key (ลองใหม่ไม่ช่วย) |
| 404 | ไม่มีหน้านี้ | ตรวจ URL |
| 429 | ขอถี่เกินไป | รอแล้วลองใหม่ |
| 500 / 503 | ฝั่งเว็บมีปัญหา | รอแล้วลองใหม่ |

## 6.2 API key ใส่ใน headers และ `raise_for_status()`

In [ ]:
no_key = requests.get(f"{API_URL}/v1/campaigns", timeout=10)
print(no_key.status_code, no_key.json())

In [ ]:
no_key.raise_for_status()      # ❌ status ไม่ใช่ 2xx -> raise HTTPError ทำให้รู้ตัวทันที

In [ ]:
headers = {"X-API-Key": API_KEY}
campaigns = requests.get(f"{API_URL}/v1/campaigns", headers=headers, timeout=10)
campaigns.raise_for_status()                    # ผ่าน = ไม่มีอะไรเกิดขึ้น
for c in campaigns.json()["data"]:
    print(c["campaign_id"], "|", c["channel"], "|", c["campaign_name"])

## 6.3 ส่งเงื่อนไขด้วย `params` และรับมือเมื่อเว็บไม่ว่าง

`params={"date": "2026-09-25"}` → requests ต่อท้าย URL ให้เป็น `?date=2026-09-25` เอง

🧪 กดรัน cell ข้างล่างซ้ำหลาย ๆ รอบ จะเห็น 429 / 503 บ้างเป็นบางครั้ง

In [ ]:
r = requests.get(f"{API_URL}/v1/ad-spend", params={"date": "2026-09-25", "page": 1}, headers=headers, timeout=10)
print(r.url)
print(r.status_code, r.text[:200])

In [ ]:
import time


def get_json(path: str, params: dict | None = None, max_tries: int = 5) -> dict:
    """ขอข้อมูลจาก API แล้วคืน dict · ลองใหม่เฉพาะเมื่อเว็บไม่ว่าง (429/5xx) · ปัญหาอื่นหยุดทันที"""
    for attempt in range(1, max_tries + 1):
        resp = requests.get(f"{API_URL}{path}", params=params, headers=headers, timeout=10)
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_tries:
            wait = int(resp.headers.get("Retry-After", 1))       # เว็บบอกว่าให้รอกี่วินาที
            print(f"  ครั้งที่ {attempt}: HTTP {resp.status_code} รอ {wait} วินาที")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()


body = get_json("/v1/ad-spend", {"date": "2026-09-25", "page": 1})
print({k: body[k] for k in ["date", "page", "page_size", "total_pages", "total_records"]})

## 6.4 ข้อมูลหลายหน้า (pagination)

API ไม่ส่งข้อมูลทั้งหมดในครั้งเดียว ต้องขอทีละหน้า `page = 1, 2, 3, ...` จนถึง `total_pages`

In [ ]:
def fetch_ad_spend(day: str) -> list[dict]:
    """ค่าโฆษณาทุกแคมเปญของวันนั้น (ขอทีละหน้าจนครบ)"""
    rows, page = [], 1
    while True:
        body = get_json("/v1/ad-spend", {"date": day, "page": page})
        rows.extend(body["data"])            # extend = เอาทุกแถวของหน้านี้ต่อท้าย rows
        if page >= body["total_pages"]:
            break
        page += 1
    print(f"{day}: ได้ {len(rows)} แถว จาก {page} หน้า")
    return rows


ad_rows = fetch_ad_spend("2026-09-25")
ad_rows[:2]

## 6.5 pandas: ตารางใน Python

**DataFrame** = ตาราง (แถว × คอลัมน์) · list ของ dict แปลงเป็นตารางได้ทันที: 1 dict = 1 แถว

In [ ]:
import pandas as pd

ads = pd.DataFrame(ad_rows)
print(ads.shape)              # (จำนวนแถว, จำนวนคอลัมน์)
print(ads.dtypes)             # ชนิดของแต่ละคอลัมน์
ads.head()

In [ ]:
print("ค่าโฆษณารวม:", ads["spend"].sum(), "บาท")
print("ช่องทาง:", ads["channel"].unique())

## 6.6 อ่านไฟล์ CSV

In [ ]:
orders = pd.read_csv("../data/marketplace/marketplace_orders_2026-09-25.csv")
print(orders.shape)
orders.head()

In [ ]:
# อ่านหลายไฟล์: หาไฟล์ -> อ่านทีละไฟล์ -> ต่อกัน
from pathlib import Path

files = sorted(Path("../data/marketplace").glob("marketplace_orders_2026-09-2*.csv"))
tables = []
for f in files:
    one_day = pd.read_csv(f)
    one_day["source_file"] = f.name              # จดไว้ว่ามาจากไฟล์ไหน (ตรวจย้อนหลังได้)
    tables.append(one_day)

all_orders = pd.concat(tables, ignore_index=True)
print(len(files), "ไฟล์ ->", all_orders.shape)
all_orders["source_file"].value_counts().sort_index()

## 6.7 เลือกคอลัมน์ · กรองแถว · เพิ่มคอลัมน์

In [ ]:
all_orders[["marketplace", "sku", "qty", "unit_price"]].head(3)          # เลือกบางคอลัมน์

In [ ]:
completed = all_orders[all_orders["status"] == "completed"]              # กรองแถว
big_orders = all_orders[(all_orders["qty"] >= 2) & (all_orders["status"] != "cancelled")]   # หลายเงื่อนไข: & และ | หรือ
print(len(all_orders), len(completed), len(big_orders))

In [ ]:
all_orders["order_datetime"] = pd.to_datetime(all_orders["order_datetime"])     # ข้อความ -> วันเวลา
all_orders["order_date"] = all_orders["order_datetime"].dt.date
all_orders["line_total"] = all_orders["qty"] * all_orders["unit_price"]          # คำนวณคอลัมน์ใหม่
all_orders[["order_datetime", "order_date", "qty", "unit_price", "line_total"]].head()

In [ ]:
# หาข้อมูลเสีย: จำนวน <= 0 (ไฟล์วันที่ 26 มีแถวแบบนี้)
all_orders[all_orders["qty"] <= 0][["source_file", "order_ref", "sku", "qty"]]

## 6.8 สรุป (groupby) และรวมตาราง (merge)

In [ ]:
good = all_orders[(all_orders["status"] != "cancelled") & (all_orders["qty"] > 0)]
daily_sales = (good.groupby(["order_date", "marketplace"], as_index=False)
                   .agg(orders=("order_ref", "nunique"), units=("qty", "sum"), revenue=("line_total", "sum"))
                   .sort_values(["order_date", "revenue"], ascending=[True, False]))
daily_sales.head(9)

In [ ]:
# merge = เอาสองตารางมาต่อกันด้วยคอลัมน์ที่ตรงกัน (เหมือน JOIN ใน SQL)
ad_by_day = ads.groupby("date", as_index=False)["spend"].sum()
ad_by_day["order_date"] = pd.to_datetime(ad_by_day["date"]).dt.date
sales_by_day = good.groupby("order_date", as_index=False)["line_total"].sum()

report = sales_by_day.merge(ad_by_day[["order_date", "spend"]], on="order_date", how="left")
report

## 6.9 บันทึกเป็นไฟล์

In [ ]:
Path("../work").mkdir(exist_ok=True)
daily_sales.to_csv("../work/daily_sales.csv", index=False)
daily_sales.to_parquet("../work/daily_sales.parquet", index=False)     # parquet: เล็ก เร็ว เก็บชนิดข้อมูลไว้ด้วย
pd.read_parquet("../work/daily_sales.parquet").dtypes

💡 **parquet** คือรูปแบบไฟล์ที่ Iceberg ใช้เก็บข้อมูลจริง (บทถัดไป)

## ✅ สรุปบทที่ 6

| เรื่อง | ตัวอย่าง |
|---|---|
| ขอข้อมูล | `requests.get(url, params={...}, headers={...}, timeout=10)` |
| ตรวจผล | `.status_code`, `.json()`, `.raise_for_status()` |
| หลายหน้า | `while True` + `page += 1` + `break` |
| สร้างตาราง | `pd.DataFrame(list_of_dicts)`, `pd.read_csv(path)`, `pd.concat([...])` |
| จัดการตาราง | `df[["a", "b"]]`, `df[df["x"] > 0]`, `df["new"] = ...` |
| สรุป / รวม | `.groupby(...).agg(...)`, `.merge(other, on=..., how="left")` |
| บันทึก | `.to_csv()`, `.to_parquet()` |

🧪 **แบบฝึกท้ายบท:** ดึงค่าโฆษณา 3 วัน (`2026-09-23` ถึง `2026-09-25`) รวมเป็นตารางเดียว
แล้วหา **ค่าโฆษณารวมต่อช่องทาง (channel)** เรียงจากมากไปน้อย

In [ ]:
# เขียนต่อตรงนี้